In [ ]:
# IMPORTS
from __future__ import annotations

import copy
import math
import os
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

In [ ]:
# DATA PREP
# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "data/raw/mid_datav4.csv"
TRAIN_FILE = "data/raw/train_data.csv"
TEST_FILE = "data/raw/test_data.csv"
RANDOM_SEED = 42

keep_cols = [
    "MMSI",
    "voyage_id",
    "num_pings",
    "row_id",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "COG",
    "HEADING",
    "dt",
]

rename_dict = {
    "PERIOD": "TIME",
    "period": "TIME",
    "time": "TIME",

    "LAT_AVG": "LAT",
    "lat_avg": "LAT",
    "lat": "LAT",

    "LON_AVG": "LON",
    "lon_avg": "LON",
    "lon": "LON",

    "SPEED_KNOTS": "SPEED",
    "speed_knots": "SPEED",
    "speed": "SPEED",

    "COG_DEG": "COG",
    "cog_deg": "COG",
    "cog": "COG",

    "HEADING_DEG": "HEADING",
    "heading_deg": "HEADING",
    "heading": "HEADING",

    "time_diff": "dt",
    "TIME_DIFF": "dt",
    "DT": "dt",

}

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(INPUT_FILE)

# ============================================================
# RENAME COLUMNS
# ============================================================

df = df.rename(columns=rename_dict)

# create row_id if missing
if "row_id" not in df.columns:
    df["row_id"] = range(len(df))

# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

missing = [col for col in keep_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ============================================================
# KEEP ONLY WANTED COLUMNS
# ============================================================

df = df[keep_cols].copy()

# ============================================================
# CLEAN ONLY THE COLUMNS THAT SHOULD BE NUMERIC
# ============================================================

df["MMSI"] = pd.to_numeric(df["MMSI"], errors="coerce")
df["num_pings"] = pd.to_numeric(df["num_pings"], errors="coerce")

df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")

df["LAT"] = pd.to_numeric(df["LAT"], errors="coerce")
df["LON"] = pd.to_numeric(df["LON"], errors="coerce")
df["SPEED"] = pd.to_numeric(df["SPEED"], errors="coerce")
df["COG"] = pd.to_numeric(df["COG"], errors="coerce")
df["HEADING"] = pd.to_numeric(df["HEADING"], errors="coerce")
df["dt"] = pd.to_numeric(df["dt"], errors="coerce")

# keep voyage_id and row_id as-is
df["voyage_id"] = df["voyage_id"].astype(str)
df["row_id"] = df["row_id"].astype(str)

# drop rows missing essential values
df = df.dropna(subset=["MMSI", "TIME", "LAT", "LON", "SPEED", "COG", "HEADING", "dt"]).copy()

# standardize remaining types
df["MMSI"] = df["MMSI"].astype("int64").astype(str)
df["num_pings"] = df["num_pings"].fillna(-1).astype("int64")

# ============================================================
# SPLIT 70/30 BY ROWS USING WHOLE MMSI GROUPS
# ============================================================

mmsi_counts = df["MMSI"].value_counts().reset_index()
mmsi_counts.columns = ["MMSI", "row_count"]
mmsi_counts = mmsi_counts.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

total_rows = len(df)
target_train_rows = 0.70 * total_rows

train_mmsis = []
running_rows = 0

for _, row in mmsi_counts.iterrows():
    if running_rows < target_train_rows:
        train_mmsis.append(row["MMSI"])
        running_rows += row["row_count"]
    else:
        break

train_mmsis = set(train_mmsis)

train_df = df[df["MMSI"].isin(train_mmsis)].copy()
test_df = df[~df["MMSI"].isin(train_mmsis)].copy()

# ============================================================
# CHECK OVERLAP
# ============================================================

overlap = set(train_df["MMSI"]).intersection(set(test_df["MMSI"]))
print("Overlap:", overlap)

# ============================================================
# REPORT
# ============================================================

print("Total rows:", total_rows)
print("Train rows:", len(train_df), f"({len(train_df)/total_rows:.2%})")
print("Test rows:", len(test_df), f"({len(test_df)/total_rows:.2%})")
print("Train MMSIs:", train_df["MMSI"].nunique())
print("Test MMSIs:", test_df["MMSI"].nunique())

# ============================================================
# SAVE
# ============================================================

train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print(f"Saved {TRAIN_FILE}")
print(f"Saved {TEST_FILE}")